<a href="https://colab.research.google.com/github/marcelalozano27-ship-it/bsan6200-assignment5/blob/main/rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5 — Option B: Job Fit Analyzer
## BSAN 6200: Text Mining & Social Media Analytics — Spring 2026

**Student Name:** Marcela Lozano
**Option:** B — Job Fit Analyzer  
**API Path:** [Paid ]  

---

### Table of Contents
1. [Setup and Imports](#1-setup)
2. [Load Job Descriptions and Resume](#2-loading)
3. [Text Chunking](#3-chunking)
4. [Embedding and Vector Store](#4-embedding)
5. [Analysis Prompts and Chain](#5-analysis)
6. [Zero-shot vs. Few-shot Comparison](#6-comparison)
7. [Evaluation](#7-evaluation)

> **Reminder:** The Streamlit app is a separate file (`streamlit_app.py`). This notebook builds and tests the analysis pipeline.  
> See the Option B Implementation Guide for detailed step requirements.

---
<a id="1-setup"></a>
## 1. Setup and Imports

Install required packages and load your API key from a `.env` file.  
**Do NOT hardcode API keys in this notebook.**

Suggested packages: `langchain`, `langchain-openai` or `langchain-community`, `chromadb` or `faiss-cpu`, `pypdf`, `python-dotenv`, `pandas`, `sentence-transformers` (free path)

In [1]:
# ── Install packages ──
!pip install -q chromadb sentence-transformers huggingface-hub python-dotenv

import os
import re
import pandas as pd
import requests
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
load_dotenv()


print("Imports working")

Imports working


---
<a id="2-loading"></a>
## 2. Load Job Descriptions and Resume

**Required:**
- 10+ JD files in `data/job_descriptions/` (each as a separate .txt or .pdf)
- Your resume in `data/resume/`
- A metadata file `data/jd_metadata.csv` with columns: filename, company, title, source_url, date_collected

Print: number of JDs loaded, number of resume docs, and preview content from each.

In [2]:
# ── Load JD metadata ──
jd_metadata = pd.read_csv(
    "https://raw.githubusercontent.com/marcelalozano27-ship-it/bsan6200-assignment5/main/data/jd_metadata.csv"
)

print(jd_metadata)

                                             filename                Company  \
0                 jd_LAClippers_data_analyst_lead.txt            LA Clippers   
1         jd_alo_production_supplyplanning_intern.txt                    ALO   
2                jd_axs_associate_product_manager.txt                    AXS   
3          jd_ephemeris_financial_strategy_intern.txt             Ephemeris    
4                       jd_fedex_analytics_intern.txt                 Fedex    
5           jd_revolve_data_analyst_merchandising.txt                Revolve   
6       jd_roku_content_analytics_insights_intern.txt                   Roku   
7       jd_skechers_strategic_partnerships_intern.txt               Skechers   
8   jd_sony_insights_research_analytics_summerinte...                   Sony   
9      jd_tiktok_strategic_partner_manager_intern.txt                 TikTok   
10         jd_tinder_corporate_rotational_analyst.txt                 Tinder   
11  jd_universalmusicgroup_Brand_label_o

In [3]:
# ── Load JD documents and resume ──
from langchain_core.documents import Document
import requests

jd_documents = []

base_url = "https://raw.githubusercontent.com/marcelalozano27-ship-it/bsan6200-assignment5/main/data/job_descriptions/"

for _, row in jd_metadata.iterrows():
    filename = row["filename"]
    url = base_url + filename

    text = requests.get(url).text

    doc = Document(
        page_content=text,
        metadata={
            "filename": filename,
            "company": row.get("Company", ""),
            "title": row.get("title", ""),
            "source_url": row.get("source_url", ""),
            "date_collected": row.get("date_collected", ""),
            "doc_type": "job_description"
        }
    )

    jd_documents.append(doc)

print("Loaded job descriptions:", len(jd_documents))
print(jd_documents[0].metadata)
print(jd_documents[0].page_content[:500])

Loaded job descriptions: 13
{'filename': 'jd_LAClippers_data_analyst_lead.txt', 'company': 'LA Clippers', 'title': 'Data Analyst Lead', 'source_url': 'https://www.nba.com/clippers/company/careers/openpositionlaclippers?gh_jid=4651394006&gh_src=9f5ba8c96us', 'date_collected': '4/27/2026', 'doc_type': 'job_description'}

The LA Clippers are looking to hire a Data Analyst Lead to drive data‑informed decisions across ticketing strategy across pricing, inventory management, and demand generation for our business. This role owns the analytics that power everyday decisions—from forecasting and performance reporting to optimization recommendations that increase conversion and yield. You will partner closely with Marketing, Sales and Finance stakeholders to translate data into clear actions and measurable outcomes.

T


In [4]:
resume_url = "https://raw.githubusercontent.com/marcelalozano27-ship-it/bsan6200-assignment5/main/data/resume/resume.txt"

resume_text = requests.get(resume_url).text

resume_doc = Document(
    page_content=resume_text,
    metadata={
        "filename": "resume.txt",
        "doc_type": "resume"
    }
)

print("Loaded resume")
print(resume_doc.page_content[:500])

Loaded resume
MARCELA LOZANO
Los Angeles, CA
marcelalozano27@gmail.com
linkedin.com/in/marcelalozano

EDUCATION
Loyola Marymount University
M.S. Business Analytics, Expected Aug 2026
GPA: 4.0

University of Texas at Austin
Certification: Data Science and Data Management Systems
Skills: SQL, Python, Tableau, Google Cloud Platform, Power BI, Microsoft Excel

Loyola Marymount University
B.A. Economics, Minor Spanish
Magna Cum Laude, GPA: 3.89
Valedictorian of Economics Major

TECHNICAL SKILLS
Python
SQL
Tableau



In [5]:
# ── Preview sample content ──
all_documents = jd_documents + [resume_doc]

print("Total documents including Resume:", len(all_documents))
for i in range(3):
    print(f"\n--- Job Description {i} ---")
    print(jd_documents[i].metadata)
    print(jd_documents[i].page_content[:300])

Total documents including Resume: 14

--- Job Description 0 ---
{'filename': 'jd_LAClippers_data_analyst_lead.txt', 'company': 'LA Clippers', 'title': 'Data Analyst Lead', 'source_url': 'https://www.nba.com/clippers/company/careers/openpositionlaclippers?gh_jid=4651394006&gh_src=9f5ba8c96us', 'date_collected': '4/27/2026', 'doc_type': 'job_description'}

The LA Clippers are looking to hire a Data Analyst Lead to drive data‑informed decisions across ticketing strategy across pricing, inventory management, and demand generation for our business. This role owns the analytics that power everyday decisions—from forecasting and performance reporting to o

--- Job Description 1 ---
{'filename': 'jd_alo_production_supplyplanning_intern.txt', 'company': 'ALO', 'title': 'Production & Supply Planning Intern', 'source_url': 'https://www.aloyoga.com/pages/careers', 'date_collected': '4/26/2026', 'doc_type': 'job_description'}

We are seeking a motivated and detail-oriented Production & Supply Plann

The documents pulled from GitHub contain 13 job descriptions and 1 resume.

---
<a id="3-chunking"></a>
## 3. Text Chunking

Splitting documents into chunks using 2 chunking strategies. Fixed size chunking and sentence aware chunking were compared quantitatively.

In [6]:
# ── Strategy 1 ── Fixed size chunking

def chunk_text_fixed(text, chunk_size=800, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if len(chunk) > 50:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


fixed_chunks = []

for doc in all_documents:
    chunks = chunk_text_fixed(doc.page_content, chunk_size=800, overlap=100)

    for i, chunk in enumerate(chunks):
        fixed_chunks.append({
            "text": chunk,
            "filename": doc.metadata.get("filename", ""),
            "company": doc.metadata.get("company", ""),
            "title": doc.metadata.get("title", ""),
            "doc_type": doc.metadata.get("doc_type", ""),
            "chunk_id": i,
            "chunk_strategy": "fixed_size"
        })

print("Strategy 1 chunks:", len(fixed_chunks))
print(fixed_chunks[0]["text"][:300])

Strategy 1 chunks: 56
The LA Clippers are looking to hire a Data Analyst Lead to drive data‑informed decisions across ticketing strategy across pricing, inventory management, and demand generation for our business. This role owns the analytics that power everyday decisions—from forecasting and performance reporting to op


In [7]:
# ── Strategy 2 ──
def chunk_text_by_sentences(text, chunk_size=800, overlap_words=20):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) > chunk_size and current_chunk:
            chunks.append(current_chunk.strip())

            words = current_chunk.split()
            overlap_text = " ".join(words[-overlap_words:]) if len(words) > overlap_words else current_chunk
            current_chunk = overlap_text + " " + sentence
        else:
            current_chunk += (" " if current_chunk else "") + sentence

    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return [c for c in chunks if len(c) > 50]


sentence_chunks = []

for doc in all_documents:
    chunks = chunk_text_by_sentences(doc.page_content, chunk_size=800, overlap_words=20)

    for i, chunk in enumerate(chunks):
        sentence_chunks.append({
            "text": chunk,
            "filename": doc.metadata.get("filename", ""),
            "company": doc.metadata.get("company", ""),
            "title": doc.metadata.get("title", ""),
            "doc_type": doc.metadata.get("doc_type", ""),
            "chunk_id": i,
            "chunk_strategy": "sentence_aware"
        })

print("Strategy 2 chunks:", len(sentence_chunks))
print(sentence_chunks[0]["text"][:300])

Strategy 2 chunks: 51
The LA Clippers are looking to hire a Data Analyst Lead to drive data‑informed decisions across ticketing strategy across pricing, inventory management, and demand generation for our business. This role owns the analytics that power everyday decisions—from forecasting and performance reporting to op


In [8]:
# ── Compare strategies ──
fixed_chunk_size = 800
fixed_overlap = 100

sentence_chunk_size = 800
sentence_overlap_words = 20

comparison = pd.DataFrame({
    "strategy": ["fixed_size", "sentence_aware"],
    "chunk_size": [fixed_chunk_size, sentence_chunk_size],
    "overlap": [fixed_overlap, sentence_overlap_words],
    "overlap_unit": ["characters", "words"],
    "num_chunks": [len(fixed_chunks), len(sentence_chunks)]
})

comparison

,strategy,chunk_size,overlap,overlap_unit,num_chunks
0,fixed_size,800,100,characters,56
1,sentence_aware,800,20,words,51


### Quantitative Chunking Comparison

Strategy 1 used fixed-size chunking and produced 56 total chunks using a chunk size of 800 characters and 100 characters of overlap. Strategy 2 uses sentence-aware chunking and produced 51 total chunks using a chunk size of 800 and 20 words of overlap.

The sentence-aware strategy produced 5 fewer chunks than the fixed-size strategy, which is about an 8.9% reduction in total chunks.

The sentence-aware approach is slightly more efficient because it created fewer chunks while preserving more complete sentence-level context.

### Final Choice

I chose sentence-aware chunking as the final strategy. Even though both strategies used the same general chunk size, sentence-aware chunking is better suited for job descriptions because job requirements, responsibilities, and qualifications are usually written in complete sentences or bullet points. Preserving those complete thoughts will help the retrieval step return more meaningful sections of the job description.

The fixed-size strategy produced more chunks, but it may split important requirements across chunk boundaries. The sentence-aware strategy produced fewer chunks and better preserved the meaning of each section, making it the stronger option for the final job fit analysis.

### Chunking Decision

**Which strategy did you choose?**  
Sentence Aware Chunking

**Why?**  
I'm using sentence aware chunking to preserve complete thoughts and sentences within the job descriptions. This helps improve the quality of retrieved information. Fixed size chunking splits important requirements across chunks and is therefore less meaningful for the analysis.  

**Final settings (chunk_size, overlap):**
chunk_size = 800
overlap = 20 words

---
<a id="4-embedding"></a>
## 4. Embedding and Vector Store

I embedded the final sentence-aware chunks and stored them in a ChromaDB vector database.

**Embedding path used:** Free path  
**Embedding model:** `sentence-transformers/all-MiniLM-L6-v2`  
**Vector store:** ChromaDB  

I chose the free embedding path because it avoided paid embedding API calls while still supporting semantic similarity search across the resume and job descriptions.

After creating the store, I ran a test similarity search to verify that relevant chunks were successfully retrieved from the database.

In [ ]:
import logging
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
from sentence_transformers import SentenceTransformer
import chromadb

model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")


client = chromadb.Client()
collection = client.create_collection(name="job_fit", get_or_create=True)

# use BEST chunks
all_chunks = sentence_chunks

# create embeddings + store
texts = [chunk["text"] for chunk in all_chunks]
embeddings = model.encode(texts, show_progress_bar=False)

# add to chroma
collection.add(
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=all_chunks,
    ids=[str(i) for i in range(len(texts))]
)

print("Vector DB created with", len(texts), "chunks")

In [10]:
# ── Verify: run a test similarity search ──

results = collection.query(
    query_texts=["Python SQL data analyst skills"],
    n_results=3
)

for i, doc in enumerate(results["documents"][0]):
    print(f"\n--- Result {i+1} ---")
    print(doc[:300])


--- Result 1 ---
Strong analytical and problem-solving skills. Proficiency in Microsoft Excel and basic data analysis techniques. Excellent written and verbal communication skills. Ability to work collaboratively in a fast-paced, team-oriented environment. Ability to work full-time onsite in Long Beach, CA. Preferre

--- Result 2 ---
Science and Data Management Systems Skills: SQL, Python, Tableau, Google Cloud Platform, Power BI, Microsoft Excel Loyola Marymount University B.A. Economics, Minor Spanish
Magna Cum Laude, GPA: 3.89
Valedictorian of Economics Major

TECHNICAL SKILLS
Python
SQL
Tableau
Excel
Pandas
NumPy
scikit-lear

--- Result 3 ---
MARCELA LOZANO
Los Angeles, CA
marcelalozano27@gmail.com
linkedin.com/in/marcelalozano

EDUCATION
Loyola Marymount University
M.S. Business Analytics, Expected Aug 2026
GPA: 4.0

University of Texas at Austin
Certification: Data Science and Data Management Systems
Skills: SQL, Python, Tableau, Googl


### Similarity Search Evaluation

The similarity search successfully retrieved the relevant chunks related to analytics, SQL, Python, Tableau, Excel, and communication skills. The retrieved results included both job description requirements as well as matching resume sections. This confirms that the embedding model and ChromaDB vector store were functioning correctly.

Result 1 retrieved a job description chunk emphasizing analytical skills, Excel, communication, and teamwork requirements. Results 2 and 3 retrieved resume chunks containing technical skills such as SQL, Python, Tableau, Power BI, and educational background information. These results demonstrate that the retrieval system was able to identify semantically related content between the resume and job descriptions.

The retrieved chunks were relevant to the search query “Python SQL data analyst skills,” and the vector database successfully captured semantic similarity.

---
<a id="5-analysis"></a>
## 5. Analysis Prompts and Chain

Build 3 analysis types, each with its own prompt:

1. **Skill Gap Report:** Compare resume skills vs. JD requirements. Output matching skills, missing skills, and recommended actions.
2. **Keyword Alignment:** Extract key terms from a JD, check which appear in the resume, report a match rate.
3. **Fit Summary:** 3-4 sentence narrative assessment citing evidence from both documents.

You also need to wire up the LLM and a way to pass a specific JD + resume into each prompt.

**Required:** Document at least 3 prompt iterations total (across any analysis type) with rationale.

**Reminder:** Prompt design must be your own work (Tier 2 — AI prohibited for this step).

In [11]:
# ── Initialize LLM ──
from openai import OpenAI

# load environment variables
load_dotenv(".env")

# initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("LLM initialized")

LLM initialized


In [12]:
# ── Analysis 1: Skill Gap Report ──
def get_skill_gap_analysis(query):
    results = collection.query(
        query_texts=[query],
        n_results=8
    )

    context = "\n\n".join(results["documents"][0])

    prompt = f"""
You are a job fit analyzer. Use ONLY the provided context from the job description and resume.

Task:
Compare the job description against the candidate resume and create a Skill Gap Report.

Rules:
- Do not assume skills that are not stated in the resume.
- Do not say "assuming," "not confirmed," or "uncertain."
- Make a clear decision based only on the available text.
- Before listing a skill as missing, first check whether it appears anywhere in the resume context.
- If SQL, Python, Tableau, Excel, Pandas, NumPy, scikit-learn, or analytics experience appear in the resume, list them as matching skills, not gaps.
- Keep missing skills limited to truly absent or weakly demonstrated areas.
- When listing matching skills, cite exact evidence from the resume, such as tools, coursework, projects, or work experience.
- Provide practical recommendations for closing each gap.

Return the response in this structure:

1. Matching skills found in the resume
- Skill:
- Resume evidence:
- Why it matches the JD:

2. Required skills from the job description
- List the major required skills from the JD.

3. Missing or weak skills
- Gap:
- Why it is a gap:

4. Recommended actions to close each gap
- Gap:
- Recommended action:

Context:
{context}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You compare resumes to job descriptions using only the provided text."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [13]:
analysis = get_skill_gap_analysis(
    "resume skills SQL Python Tableau Excel candidate experience"
)
print(analysis)

1. **Matching skills found in the resume**
- **Skill:** SQL
- **Resume evidence:** "Skills: SQL, Python, Tableau, Google Cloud Platform, Power BI, Microsoft Excel"
- **Why it matches the JD:** The job description requires advanced SQL skills for querying and analyzing datasets, which is explicitly stated in the resume.

- **Skill:** Python
- **Resume evidence:** "Skills: SQL, Python, Tableau, Google Cloud Platform, Power BI, Microsoft Excel"
- **Why it matches the JD:** The job description mentions familiarity with Python as a plus, and the candidate has demonstrated proficiency in Python.

- **Skill:** Tableau
- **Resume evidence:** "Skills: SQL, Python, Tableau, Google Cloud Platform, Power BI, Microsoft Excel"
- **Why it matches the JD:** The job description requires experience with data visualization tools such as Tableau, which the candidate has listed as a skill.

- **Skill:** Excel
- **Resume evidence:** "Skills: SQL, Python, Tableau, Google Cloud Platform, Power BI, Microsoft E

In [14]:
# ── Analysis 2: Keyword Alignment ──
def get_keyword_alignment(query):
    results = collection.query(
        query_texts=[query],
        n_results=10
    )

    docs = results["documents"][0]

    # Prioritize resume chunks so the model sees your actual skills
    resume_chunks = [
        d for d in docs
        if "MARCELA" in d or "TECHNICAL SKILLS" in d or "EDUCATION" in d or "SQL" in d
    ]
    jd_chunks = [d for d in docs if d not in resume_chunks]

    context = "\n\n".join(resume_chunks + jd_chunks)

    prompt = f"""
You are a job fit analyzer. Use ONLY the provided job description and resume context.

Task:
Create a Keyword Alignment analysis comparing the job description keywords against the resume.

Instructions:
- Extract 12 to 15 important keywords or phrases from the job description.
- For each keyword, determine whether it appears directly in the resume, appears as a semantic equivalent, or is missing.
- Do not mark SQL, Python, Tableau, Excel, analytics, or data visualization as missing if they appear anywhere in the resume context.
- Calculate an overall keyword match percentage:
  match percentage = (direct matches + semantic matches) / total keywords * 100
- Be specific and evidence-based.

Return the output in this structure:

1. Keyword Alignment Table
Columns:
- JD Keyword/Phrase
- Resume Match Status: Direct Match, Semantic Match, or Missing
- Resume Evidence
- Notes

2. Overall Keyword Match Percentage
Show the calculation.

3. Brief Interpretation
Write 3 to 4 sentences explaining what the match percentage means for the candidate's fit.

Context:
{context}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You compare job description keywords to resume evidence using only provided text."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [15]:
keyword_analysis = get_keyword_alignment(
    "Data Analyst job requirements keywords SQL Python Tableau Excel analytics pricing forecasting"
)

print(keyword_analysis)

### 1. Keyword Alignment Table

| JD Keyword/Phrase                                         | Resume Match Status | Resume Evidence                                                                                                           | Notes                                                                                   |
|----------------------------------------------------------|---------------------|--------------------------------------------------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------|
| Data analysis                                            | Direct Match        | "data analysis techniques"                                                                                               | Directly mentioned in the resume.                                                      |
| SQL                                                      | Direct Match      

In [16]:
# ── Analysis 3: Fit Summary ──
def get_fit_summary(query):
    results = collection.query(
        query_texts=[query],
        n_results=8
    )

    docs = results["documents"][0]

    # prioritize resume content
    resume_chunks = [
        d for d in docs
        if "MARCELA" in d or "TECHNICAL SKILLS" in d or "EDUCATION" in d or "SQL" in d
    ]
    jd_chunks = [d for d in docs if d not in resume_chunks]

    context = "\n\n".join(resume_chunks + jd_chunks)

    prompt = f"""
You are a hiring analyst evaluating candidate fit.

Task:
Provide a clear, concise Fit Summary comparing the candidate resume to the job description.

Instructions:
- Use ONLY the provided context.
- Be decisive and do not say "uncertain" or "assuming."
- Recognize existing strengths such as SQL, Python, Tableau, Excel, analytics experience if present.
- Focus on overall fit, not just listing skills.

Return the output in this structure:

1. Overall Fit Score (0–100)
- Give a numeric score
- Brief justification

2. Strengths
- 3–5 key strengths that align with the role

3. Weaknesses / Gaps
- 2–4 meaningful gaps (focus on real gaps, not tools already present)

4. Final Hiring Recommendation
- Strong Yes, Yes, Maybe, or No
- 2–3 sentence explanation

Context:
{context}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You evaluate candidate fit using only provided resume and job description text."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [17]:
fit_summary = get_fit_summary(
    "Data Analyst job fit SQL Python Tableau Excel analytics pricing forecasting"
)

print(fit_summary)

1. Overall Fit Score: 85
- Marcela has a strong educational background in Business Analytics and relevant technical skills in SQL, Python, Tableau, and Excel. Her experience in analytics consulting and project work aligns well with the requirements of the Data Analyst Lead role, particularly in data-driven decision-making and stakeholder collaboration.

2. Strengths
- Proficient in SQL, Python, and Tableau, which are essential for data analysis and visualization.
- Experience in building ETL pipelines and performing advanced analytics, including NLP and customer segmentation, which are relevant to the role's focus on data-driven decision-making.
- Strong communication skills demonstrated through collaboration with various teams and delivering insights that influence business decisions.
- Proven ability to translate complex data into actionable insights, aligning with the need for a "data translator" in the merchandising team.

3. Weaknesses / Gaps
- Lacks direct experience in pricing, 

## Prompt Iteration Log

Document at least 3 total iterations across any of the analysis types.

**Iteration 1:**

**Analysis:** Skill Gap Report

**What changed and why?**
The initial prompt asked the model to identify missing skills, transferable skills, and recommendations based on the resume and job description. However, the output occasionally labeled skills such as SQL, Tableau, and Power BI as missing even when they were clearly listed in the resume.

To improve the accuracy, the final prompt instructed the model to verify resume evidence before identifying a skill gap. This helped reduce the number of false positives and produced more realistic gaps focused on advanced Excel, forecasting, pricing/revenue optimization, experimentation platforms, and business partnering.

**What Improved**:
The final prompt improved accuracy by forcing the model to check resume evidence before labeling a skill as missing. The output improved because it followed the resume content more closely and reduced incorrect skill gaps.


**Iteration 2:**


**Analysis:** Skill Gap Report

**What changed and why?**
The second version of the prompt added more explicit section formatting for:
- Missing Skills
- Transferable Skills
- Recommendations
The model was mixing strengths and weaknesses together which made the comparison harder to evaluate.

**What Improved**:
The responses became a lot easier to read with more structure and stronger actionable guidance. It also improved consistency across all analyzed job descriptions.


**Iteration 3:**


**Analysis:** Skill Gap Report

**What changed and why?**
I added that years of experience should not be assumed unless it is explicitly stated in the resume. The earlier versions were inferring experience levels or overstating them.

**What Improved**:
The final outputs became more grounded in the source documents and reduced hallucinated assumptions about experience level and technical depth.

---
<a id="6-comparison"></a>
## 6. Zero-shot vs. Few-shot Comparison


For this comparison I tested the Skill Gap Analysis prompt using both zero-shot and few-shot prompting. The zero-shot prompt asked the model to identify matched skills, missing skills, and recommendations without giving an example. The few-shot prompt included example input/output pairs to guide the model's response structure.

In [18]:
# ── Few-shot version of your Skill Gap analysis ──


few_shot_prompt = """
You are an AI career assistant.

Example 1:
Resume:
- Experience with Python, SQL, Tableau, and client analytics
- Worked with cross functional teams
- Built dashboards and reports

Job Description:
- Requires Python, SQL, Power BI, and machine learning experience

Output:
Missing Skills:
- Power BI
- Machine Learning

Transferable Skills:
- Tableau experience demonstrates data visualization background
- Analytics reporting experience transfers well to BI tasks

Recommendation:
Candidate should highlight analytical project experience and mention any background or exposure to predictive modeling.


Example 2:
Resume:
- Experience with Excel, operations, and customer management
- Conducted research and business reporting

Job Description:
- Requires stakeholder communication, data analysis, and dashboard reporting

Output:
Missing Skills:
- Dashboard software specifically mentioned in the JD

Transferable Skills:
- Research and reporting experience aligns with analytical communication requirements
- Customer management supports stakeholder collaboration

Recommendation:
Candidate should emphasize reporting tools and analytical communication experience.


Now analyze the following resume and job description.

Resume:
{resume}

Job Description:
{job_description}

Provide:
1. Missing Skills
2. Transferable Skills
3. Recommendations
"""

In [19]:
# ── Run both on the same JD, display side by side ──


sample_jd = jd_documents[0].page_content
sample_resume = resume_doc.page_content

# Zero-shot
zero_shot_prompt = f"""
Analyze the candidate's skill gaps compared to the job description.

Resume:
{sample_resume}

Job Description:
{sample_jd}

Provide:
1. Missing Skills
2. Transferable Skills
3. Recommendations
"""

zero_shot_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You compare resumes to job descriptions using only the provided text."
        },
        {
            "role": "user",
            "content": zero_shot_prompt
        }
    ],
    temperature=0.2
)

# Few-shot
formatted_few_shot = few_shot_prompt.format(
    resume=sample_resume,
    job_description=sample_jd
)

few_shot_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You are an AI career assistant."
        },
        {
            "role": "user",
            "content": formatted_few_shot
        }
    ],
    temperature=0.2
)

print("========== ZERO-SHOT OUTPUT ==========\n")
print(zero_shot_response.choices[0].message.content)

print("\n\n========== FEW-SHOT OUTPUT ==========\n")
print(few_shot_response.choices[0].message.content)

========== ZERO-SHOT OUTPUT ==========

### 1. Missing Skills
- **Experience Level**: The job requires 5–7+ years of analytics experience, while the candidate has less than 5 years of experience.
- **Pricing Strategy and Elasticity Analysis**: The candidate's resume does not explicitly mention experience with pricing strategy or elasticity analysis, which is crucial for the role.
- **Advanced SQL and BI Skills**: While the candidate lists SQL and BI tools like Tableau and Power BI, the job description emphasizes advanced SQL skills and proven ability to model data and tell a clear story with visuals. The candidate's experience in this area is not clearly demonstrated.
- **Hands-on ML Experience**: The job requires hands-on experience with regression/forecasting and propensity/uplift modeling. The candidate has experience with machine learning but does not specify experience in these particular areas.
- **Business Partnering Ability**: The job emphasizes strong business partnering skill

### Zero-shot vs. Few-shot Analysis

**Which analysis type did you compare?**
I compared the Skill Gap Analysis using zero-shot prompt and few-shot prompt.

**Which performed better?**

The few shot analysis performed the best overall.

**Why? (use specific examples from the outputs above)**

The output from the zero-shot analysis was more detailed but it was not necessarily better than the few shot. It inferred experience levels and provided more generalized technical gaps without actually referencing the context from the provided resume. It also was more repetitive in the recommendations it provided for the candidate.

On the other hand, the few-shot prompt had more structured and focused results. The examples provided in the prompt helped guide the model towards producing an output in the level of detail expected. The following are examples of where the few-shot version connected the job descriptions more specifically to the experience listed in the resume:
- "Experience at Coleman Research working with strategy...": In this example the few shot model connected the experience in the resume to "collaboration with cross functional teams". (The zero shot model gave a more general description saying Marcela has experience in presenting analytical findings.
- "Quantify contributions", "Emphasize her SQL and BI skills in the context of pricing and revenue optimization", "Marcela should prepare examples that showcase her ability to influence decisions through data": These are three examples where the few shot analysis provided specific and actionable recommendations for a stronger resume

Overall, the few-shot model provided more specific regarding the resume uploaded as opposed to relying on more general feedback for a candidate applying to the role. The zero-shot model provided more general feedback like "network with professionals in the field" or "consider further education". This feedback is not as helpful for signaling the actions to be taken.
The few-shot prompt performed better overall because the example input/output pairs helped the model understand the expected structure and level of detail. The few-shot response was more specific, less repetitive, easier to evaluate, and more useful for resume revision.

---
<a id="7-evaluation"></a>
## 7. Evaluation

Run all 3 analysis types on your **top 3 target JDs** (9 total analyses).

For each, score:
- **Retrieval relevance:** Did it pull the right JD sections? (Yes/Partial/No)
- **Skill identification accuracy:** Are identified skills/gaps correct? (count correct vs. incorrect)
- **Actionability:** Are recommendations specific and useful? (1-5)
- **Faithfulness:** Does output stick to document content? (Faithful/Partial/Hallucinated)



In [20]:
# __ Run 9 analyses (3 JDs x 3 analysis types) __

top_jds = jd_documents[:3]

evaluation_results = []

for i, jd in enumerate(top_jds):

    jd_text = jd.page_content
    jd_name = jd.metadata.get("title", f"JD {i+1}")

    # ---------- Fit Summary ----------
    fit_prompt = f"""
    Analyze the candidate's overall fit for this role.

    Resume:
    {sample_resume}

    Job Description:
    {jd_text}

    Provide:
    1. Overall Fit
    2. Strengths
    3. Weaknesses
    """

    fit_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a hiring analyst evaluating candidate fit."
            },
            {
                "role": "user",
                "content": fit_prompt
            }
        ],
        temperature=0.2
    )

    evaluation_results.append({
        "JD": jd_name,
        "Analysis": "Fit Summary",
        "Output": fit_response.choices[0].message.content
    })

    # ---------- Keyword Alignment ----------
    keyword_prompt = f"""
    Compare the resume and job description.

    Resume:
    {sample_resume}

    Job Description:
    {jd_text}

    Provide:
    1. Matching Keywords
    2. Missing Keywords
    3. Important Technical Skills
    """

    keyword_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a job fit analyzer comparing keywords."
            },
            {
                "role": "user",
                "content": keyword_prompt
            }
        ],
        temperature=0.2
    )

    evaluation_results.append({
        "JD": jd_name,
        "Analysis": "Keyword Alignment",
        "Output": keyword_response.choices[0].message.content
    })

    # ---------- Skill Gap Analysis ----------
    skill_gap_prompt = f"""
    Analyze the candidate's skill gaps compared to the job description.

    Resume:
    {sample_resume}

    Job Description:
    {jd_text}

    Do not assume years of experience unless explicitly stated.

    Provide:
    1. Missing Skills
    2. Transferable Skills
    3. Recommendations
    """

    skill_gap_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a job fit analyzer creating a skill gap report."
            },
            {
                "role": "user",
                "content": skill_gap_prompt
            }
        ],
        temperature=0.2
    )

    evaluation_results.append({
        "JD": jd_name,
        "Analysis": "Skill Gap Analysis",
        "Output": skill_gap_response.choices[0].message.content
    })

print(f"Completed {len(evaluation_results)} analyses.")

Completed 9 analyses.


In [21]:
# ── Summarize evaluation results ──

for result in evaluation_results:

    print("=" * 80)
    print(f"JD: {result['JD']}")
    print(f"Analysis Type: {result['Analysis']}")
    print("-" * 80)
    print(result["Output"])
    print("\n")

JD: Data Analyst Lead
Analysis Type: Fit Summary
--------------------------------------------------------------------------------
### 1. Overall Fit
Marcela Lozano appears to be a strong candidate for the Data Analyst Lead position with the LA Clippers. Her educational background in Business Analytics and Economics, combined with her technical skills in data analysis, machine learning, and data visualization tools, aligns well with the requirements of the role. Her experience in strategy and analytics consulting, particularly in developing commercialization strategies and conducting customer segmentation analysis, demonstrates her ability to drive data-informed decisions. Additionally, her hands-on experience with A/B testing and performance reporting complements the job's emphasis on optimizing pricing and promotions.

### 2. Strengths
- **Educational Background**: Marcela holds a Master's in Business Analytics (in progress) and a Bachelor's in Economics, which provides her with a sol

###Evaluation Metrics
**Actionability Rating Scale:**
Actionability was rated on a 1-5 scale based on how useful the AI output was for helping a candidate improve their resume or job application strategy.

1 = not actionable, meaning the response was vague, generic, or unrelated.

2 = slightly actionable, meaning it identified some relevant points but gave limited guidance.

3 = moderately actionable, meaning it gave useful but broad suggestions.

4 = actionable, meaning the recommendations were relevant, practical, and usable.

5 = highly actionable, meaning the response gave specific, realistic, and directly applicable recommendations tied to the resume and job description.

**Retrieval relevance** was assessed based on whether the analysis output reflected the correct job description sections and resume evidence for each role.

### Evaluation Analysis


| Position | Analysis Type | Retrieval Relevance | Skill Identification Accuracy | Actionability | Faithfulness | Notes |
|---|---|---|---|---|---|---|
| Data Analyst Lead | Fit Summary | Yes | Correct: analytics background, SQL, Python, Tableau, Power BI and communication. Partial: demand forecasting and pricing optimization slightly overstated. Incorrect: A/B Testing is listed as hands on experience but resume specifies only a conceptual understanding. | 3 | Partial | Strong overall fit assessment, but slightly overstated experience level. This analysis was overall less useful and less actionable without concrete steps. |
| Data Analyst Lead | Keyword Alignment | Yes | Correct: Tableau, Power BI, forecasting, customer segmentation, SQL, ML, communication. Partial: pricing strategy and business partnering were inferred from transferable experience. Incorrect: No major errors. | 4 | Faithful | Strong keyword extraction and alignment quality. Actionable summary is provided indicating a lack of specific experience like ticketing strategy. |
| Data Analyst Lead | Skill Gap Analysis | Yes | Correct: pricing elasticity gap, ticketing analytics gap, experimentation platforms, advanced SQL/BI skills. Partial: business partnering and ML specialization partially inferred from experience. Incorrect:  No major errors. | 5 | Faithful | Most actionable and detailed output. Correctly identified that A/B testing is only known conceptually. Links transferrable skills correctly and provides specific recommendations|
| Production & Supply Planning Intern | Fit Summary | Partial | Correct: analytical skills, data management, organization, project management. Partial: supply chain process improvement potential inferred from analytics background. Incorrect: No major errors. | 4 | Partial | Transferable skills identified correctly, but domain alignment was weaker. The model assumes that the individual would not be a good fit for this role because they are overqualified. |
| Production & Supply Planning Intern | Keyword Alignment | Yes | Correct: communication, organization, attention to detail, data management. Partial: vendor communication partially inferred from client management. Incorrect: No major errors. | 4 | Faithful | Accurate identification of missing supply chain terminology. Correctly identified that the role may not be the best fit without supply chain and fashion business experience. |
| Production & Supply Planning Intern | Skill Gap Analysis | Yes | Correct: ERP systems, vendor communication, fashion business knowledge, supply chain gaps. Partial: Excel gap somewhat overstated because Excel already exists in resume. Incorrect:  No major errors. | 5 | Faithful | Recommendations were realistic and actionable. Recommendations identify the gaps and provide straightforward and easy to follow interpretation. |
| Associate Product Manager | Fit Summary | Yes | Correct: cross-functional collaboration, analytics, communication, project management. Partial: product strategy and customer research partially inferred from consulting experience. Incorrect: No major errors. | 3 | Faithful | Strong transferable skills identified for PM role. No real actions are recommended which makes this analysis weaker. |
| Associate Product Manager | Keyword Alignment | Yes | Correct: research, collaboration, communication, data analysis, multiple projects. Partial: product knowledge partially inferred from strategy work. Incorrect: No major errors. | 4 | Faithful | Good distinction between matched and missing PM terminology. |
| Associate Product Manager | Skill Gap Analysis | Yes | Correct: product roadmap, Jira/Trello, product marketing, customer feedback gaps. Partial: PM methodology gaps inferred indirectly. Incorrect: No major errors. | 5 | Faithful | Clear recommendations for transitioning into PM roles. |



### Which analysis type worked best?

The Skill Gap Analysis performed the best overall since it produced the most specific, realistic, and actionable feedback. When compared to the Fit Summary and Keyword Alignment outputs, the Skill Gap Analysis was better at identifying what was missing from the resume and explaining how the candidate could improve their alignment with each role.

For example, the Skill Gap Analysis correctly identified gaps like pricing elasticity, ticketing analytics, ERP systems, vendor communication, fashion business knowledge, product roadmap experience, Jira/Trello, product marketing, and customer feedback experience. These outputs were also the most useful and actionable because they gave clear recommendations that could actually be applied directly to resume revisions or interview preparation.

The Fit Summary was useful for a quick overview, but it often times overstated experience. The Keyword Alignment was strong for identifying matched and missing terms but it was still less detailed than the Skill Gap Analysis.

---

### Which job descriptions produced the best and worst results? Why?

The Data Analyst Lead and Associate Product Manager job descriptions produced the strongest results. These roles aligned the best with the candidate resume because they emphasized analytics, communication, SQL, Python, Tableau, Power BI, project work, research, and cross-functional collaboration. Since these skills were clearly represented in the resume, the system was able to produce more relevant and faithful outputs.

The Production & Supply Planning Intern job description produced the weakest results. Even though the output was useful, the role was more supply chain specific while the resume was stronger in analytics, client management, and business strategy. Due to the weaker domain alignment, the model relied more on transferable skills. One example is that it connected client management to vendor communication and analytics work to supply chain process improvement. This connection is more indirect and a result of the model inferring skills.

Overall, job descriptions with strong overlap in analytics and business skills produced better results, while job descriptions requiring specialized domain experience produced more partial or inferred findings.


---

### Where did the system hallucinate or produce inaccurate results?

The system did not have any major hallucinations, but it did sometimes overstate or infer experience beyond what was directly supported by the resume. The main issue was that the model sometimes treated transferable or conceptual experience as direct experience.

For example, in the Data Analyst Lead Fit Summary, the model listed A/B testing as hands-on experience, but the resume only supports a conceptual understanding of A/B testing. The model also slightly overstated demand forecasting, pricing optimization, and pricing strategy experience. These topics may be related to analytics coursework or transferable business experience, but they were not strongly supported as direct work experience.

For the Production & Supply Planning Intern role, the system partially inferred supply chain process improvement and vendor communication from broader analytics and client management experience. These were reasonable connections, but they were not directly stated in the resume. The Excel gap was also somewhat overstated because Excel already appears in the resume.

These issues show that the system was mostly faithful, but it sometimes needed stronger instructions to separate direct evidence from inferred transferable skills.


---

### What would you improve?

I would improve the system by making the prompts more strict about evidence checking. Specifically, I would require the model to label each skill as either directly supported, partially supported, or not supported by the resume. This would reduce the chance of overstating experience.

I would also improve the retrieval step by making sure the model pulls both the most relevant job description sections and the most relevant resume sections before generating the analysis. This would make the outputs more grounded and easier to evaluate.

Another improvement would be to add a stronger hallucination check at the end of each output. The model could ideally review its own answer and flag any claims that are not directly supported by the resume or job description. This would be especially helpful for roles with weaker alignment like the Production & Supply Planning Intern position.


---

## Next Steps

1. Build your Streamlit app (`streamlit_app.py`) using the pipeline from this notebook
2. Write your Technical Manager Memo (`memo.md`)
3. Complete your AI Usage Log (`ai_log.md`)
4. Verify GitHub repository structure and commit count

---
*BSAN 6200 | Spring 2026 | Assignment 5 — Option B*